# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tannusaini2110-spec/Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.
print("Finding 1: Growing pages (74.8K, rising impressions) are 37.6% longer")
print("(3.2K vs 2.3K words) and 20% younger (184d vs 230d avg age) than")
print("declining pages (45.6K, falling impressions). [FlyRank paper, Finding #1, p.6]")
print()
print("My methodology question: This is a cross-sectional comparison of two")
print("cohorts, not a controlled test -- and the paper's own correlation matrix")
print("shows age and word count are already correlated (r = -0.517). Since younger")
print("pages are systematically longer in this portfolio, how do we know word count")
print("predicts growth independently, rather than just riding along with age? What")
print("would this comparison look like if word count were compared within age-matched")
print("bins instead of pooled across all ages?")
print()
print("Finding 2: A logistic regression (71% holdout accuracy) finds content age,")
print("days-since-update, and days-visible are the strongest predictors of growth")
print("vs. decline. [FlyRank paper, ML Appendix -- Growth & Classification, p.29]")
print()
print("My methodology question: This is the same prediction task I'm building")
print("(declining vs. non-declining pages), so the split matters a lot. The paper")
print("doesn't say whether the 80/20 holdout was a naive random split or grouped by")
print("brand/client. My own Section 2 audit found a 0.016-point gap between a naive")
print("random split (0.847) and an honest client-grouped split (0.831) on my data.")
print("Was FlyRank's 71% validated with a client-grouped holdout, or a naive random")
print("split -- and if naive, how much might it be inflated given only 57 brands")
print("feed 341K pages, leaving real room for cross-client leakage?")

Finding 1: Growing pages (74.8K, rising impressions) are 37.6% longer
(3.2K vs 2.3K words) and 20% younger (184d vs 230d avg age) than
declining pages (45.6K, falling impressions). [FlyRank paper, Finding #1, p.6]

My methodology question: This is a cross-sectional comparison of two
cohorts, not a controlled test -- and the paper's own correlation matrix
shows age and word count are already correlated (r = -0.517). Since younger
pages are systematically longer in this portfolio, how do we know word count
predicts growth independently, rather than just riding along with age? What
would this comparison look like if word count were compared within age-matched
bins instead of pooled across all ages?

Finding 2: A logistic regression (71% holdout accuracy) finds content age,
days-since-update, and days-visible are the strongest predictors of growth
vs. decline. [FlyRank paper, ML Appendix -- Growth & Classification, p.29]

My methodology question: This is the same prediction task I'm buildi

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub scikit-learn

import duckdb
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

feat = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           AVG(gsc_avg_position) as avg_position,
           SUM(gsc_impressions) as total_impressions,
           SUM(gsc_clicks) as total_clicks,
           AVG(ga4_engaged_sessions) as avg_engaged_sessions,
           SUM(ga4_pageviews) as total_pageviews,
           COUNT(*) as days_seen
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df().fillna(0)

feat["is_declining"] = (feat["total_clicks"] <= feat["total_clicks"].quantile(0.30)).astype(int)
feature_cols = ["avg_position", "total_impressions", "avg_engaged_sessions", "total_pageviews", "days_seen"]
X, y = feat[feature_cols], feat["is_declining"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model_naive = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
acc_naive = accuracy_score(y_te, model_naive.predict(X_te))

splitter = GroupShuffleSplit(test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feat, groups=feat["client_hash_id"]))
X_tr2, X_te2 = X.iloc[train_idx], X.iloc[test_idx]
y_tr2, y_te2 = y.iloc[train_idx], y.iloc[test_idx]
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
acc_honest = accuracy_score(y_te2, model_honest.predict(X_te2))

print(f"BEFORE (naive random split):    {acc_naive:.3f}")
print(f"AFTER  (honest client-grouped): {acc_honest:.3f}")
print(f"\nGap: {acc_naive - acc_honest:.3f} -- this is the inflation a naive")
print("split would have hidden.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BEFORE (naive random split):    0.845
AFTER  (honest client-grouped): 0.832

Gap: 0.013 -- this is the inflation a naive
split would have hidden.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
X_leaky = feat[feature_cols + ["total_clicks"]]
model_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_leaky.fit(X_leaky.iloc[train_idx], y_tr2)
acc_leaky = accuracy_score(y_te2, model_leaky.predict(X_leaky.iloc[test_idx]))

print(f"Honest score: {acc_honest:.3f}")
print(f"Leaky score (total_clicks included, label-derived): {acc_leaky:.3f}")
print()
print("Confirmed: total_clicks directly defines is_declining, so including it")
print("is leakage. It is correctly excluded from the final feature set.")
print()
print("No future-window data used -- all features come only from March 2026,")
print("the same month as the label window.")

Honest score: 0.832
Leaky score (total_clicks included, label-derived): 1.000

Confirmed: total_clicks directly defines is_declining, so including it
is leakage. It is correctly excluded from the final feature set.

No future-window data used -- all features come only from March 2026,
the same month as the label window.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Original claim (too strong): 'This model predicts which pages will decline.'")
print()
print("Rewritten (safe language): 'This model's honest, client-grouped test")
print(f"accuracy is {acc_honest:.1%}, observed on March 2026 data. It offers")
print("directional decision-support for prioritizing review, not a guarantee")
print("of future outcomes for any individual page.'")

Original claim (too strong): 'This model predicts which pages will decline.'

Rewritten (safe language): 'This model's honest, client-grouped test
accuracy is 83.2%, observed on March 2026 data. It offers
directional decision-support for prioritizing review, not a guarantee
of future outcomes for any individual page.'


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.